In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.animation as animation
import xarray as xr

import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from neuralop.models import UNO

torch.serialization.add_safe_globals([torch.nn.functional.gelu])

In [2]:
CURRENT_DIRECTORY = f'horizontal_correct/'
DATA_PATH = f'data'
ASSETS_PATH = f'assets'
MODELS_PATH = f'models'

filepaths = [f'{DATA_PATH}/source_{i}.hdf5' for i in range(1, 4)]

dim_map = {
    'phony_dim_0' : 'time',
    'phony_dim_1' : 'sample',
    'phony_dim_2' : 'channel',
    'phony_dim_3' : 'Y',
    'phony_dim_4' : 'X'
}

In [3]:
def preprocess(path):
    print(f'Opening dataset : {path}')
    ds = xr.open_dataset(path, engine='h5netcdf', phony_dims='sort').rename(dim_map)

    pressure = ds.dataset.isel(channel=0)
    source = ds.source

    return xr.Dataset({'pressure': pressure, 'source':source}, attrs=ds.attrs)

processed_datasets = [preprocess(f) for f in filepaths]

final_ds = xr.concat(processed_datasets, dim='sample')

final_ds = final_ds.squeeze('phony_dim_5')

Opening dataset : data/source_1.hdf5
Opening dataset : data/source_2.hdf5
Opening dataset : data/source_3.hdf5


In [ ]:
raw_p = final_ds.pressure.values
mask_2d = ~np.isnan(raw_p[0, 0])

p_min, p_max = np.nanmin(raw_p), np.nanmax(raw_p)
norm_p = np.zeros_like(raw_p, dtype='float32')
norm_p[:, :, mask_2d] = (raw_p[:, :, mask_2d] - p_min) / (p_max - p_min)

s_raw = final_ds.source.values.copy()

s_raw[..., ~mask_2d] = 0 
s_abs_max = np.max(np.abs(s_raw))
norm_s = (s_raw / s_abs_max).astype('float32') if s_abs_max != 0 else s_raw

In [ ]:
P_tensor = torch.from_numpy(norm_p).permute(1, 0, 2, 3).float()
S_tensor = torch.from_numpy(norm_s).float()

X_final = torch.stack([P_tensor, S_tensor], dim=2)

STATIC_MASK = torch.from_numpy(mask_2d.astype('float32'))

print(f"X_final shape: {X_final.shape}")

X_final shape: torch.Size([3000, 21, 2, 113, 134])


In [6]:
def get_indices(size, train_ratio=0.8, val_ratio=0.1, test_ratio=0.1):
    np.random.seed(2318)
    indices = np.arange(size)
    np.random.shuffle(indices)

    train_end = int(size * train_ratio)
    val_end = train_end + int(size * val_ratio)
    
    train_idx = indices[:train_end]
    val_idx = indices[train_end:val_end]
    test_idx = indices[val_end:]

    print(f"Dataset: Train={len(train_idx)}, Val={len(val_idx)}, Test={len(test_idx)}")

    return train_idx, val_idx, test_idx

In [8]:
class AutoRegressiveDataset(Dataset):
    def __init__(self, data, indices):
        self.data = data[indices] 
        self.N_samples = self.data.shape[0]
        self.T_steps = self.data.shape[1]
        self.samples_per_case = self.T_steps - 3 

    def __len__(self):
        return self.N_samples * self.samples_per_case

    def __getitem__(self, idx):
        s_idx = idx // self.samples_per_case
        t_idx = (idx % self.samples_per_case) + 1 

        x = torch.stack([
            self.data[s_idx, t_idx, 0],     # P_t
            self.data[s_idx, t_idx + 1, 1]  # S_t+1
        ], dim=0)
        
        y = self.data[s_idx, t_idx + 1, 0].unsqueeze(0)
        
        s_next = self.data[s_idx, t_idx + 2, 1].unsqueeze(0)
        y_next = self.data[s_idx, t_idx + 2, 0].unsqueeze(0)
        
        return x, y, s_next, y_next

In [11]:
def train_model(modes, scalings, out_channels, train_loader, val_loader, device, static_mask, epochs=250):
    model = UNO(
        in_channels=2, 
        out_channels=1,
        domain_padding=0.2,
        non_linearity=torch.nn.functional.gelu,
        hidden_channels=128,
        n_layers=4,
        projection_channels=128,
        uno_out_channels=out_channels,
        uno_n_modes=modes,
        uno_scalings=scalings,
        channel_mlp_skip="linear"
    ).to(device)

    criterion = nn.MSELoss()
    
    optimizer = optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-2) 
    mask = static_mask.to(device).float()
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=15)
    
    best_val_l2 = float('inf')
    best_model_path = "uno_aboba_fix4_1cycle.pth"
    
    patience_counter = 0
    early_stop_patience = 50 

    train_losses, val_losses_mean = [], []

    print(f"=== START PUSHFORWARD TRAINING | Device: {device} ===")

    for epoch in range(epochs):
        model.train()
        total_train_loss = 0.0

        for batch in train_loader:

            x_in, y_true1, s_next, y_true2 = [b.to(device) for b in batch]
            optimizer.zero_grad()
            
            pred1 = model(x_in)
            loss1 = criterion(pred1[:, 0] * mask, y_true1[:, 0] * mask)
            
            x_push = torch.cat([pred1, s_next], dim=1) 
            pred2 = model(x_push)
            loss2 = criterion(pred2[:, 0] * mask, y_true2[:, 0] * mask)
            
            loss = loss1 + loss2 
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            
            total_train_loss += loss.item() * x_in.size(0)
            
        epoch_train_loss = total_train_loss / len(train_loader.dataset)
        train_losses.append(epoch_train_loss)

        model.eval()
        val_l2_errors = []
        with torch.no_grad():
            for batch_v in val_loader:
                x_v, y_v, _, _ = [b.to(device) for b in batch_v]
                
                u_pred_v = model(x_v)[:, 0, :, :] * mask
                u_true_v = y_v[:, 0, :, :] * mask
                
                preds_flat = u_pred_v.reshape(x_v.size(0), -1)
                trues_flat = u_true_v.reshape(x_v.size(0), -1)
                
                diff_norm = torch.norm(preds_flat - trues_flat, p=2, dim=1)
                true_norm = torch.norm(trues_flat, p=2, dim=1)
                
                rel_l2 = (diff_norm / torch.clamp(true_norm, min=1e-8)) * 100
                val_l2_errors.extend(rel_l2.cpu().numpy())

        epoch_mean = np.mean(val_l2_errors)
        val_losses_mean.append(epoch_mean)

        scheduler.step(epoch_mean) 
        current_lr = optimizer.param_groups[0]['lr']
        
        if epoch_mean < best_val_l2:
            best_val_l2 = epoch_mean
            torch.save(model.state_dict(), best_model_path)
            status = "<--- save"
            patience_counter = 0
        else:
            status = f"[Wait {patience_counter+1}]"
            patience_counter += 1
            
        print(f"Epoch {epoch+1:03d} | LR: {current_lr:.2e} | Train Loss: {epoch_train_loss:.6f} | Val L2: {epoch_mean:.2f}% {status}")

        if patience_counter >= early_stop_patience:
            print("=== EARLY STOPPING ===")
            break

    model.load_state_dict(torch.load(best_model_path, map_location=device))
    return model, train_losses, val_losses_mean, []

In [12]:
device = "cuda:1" if torch.cuda.is_available() else "cpu"
MODES = [[32, 32], [24, 24], [24, 24], [32, 32]]
SCALINGS = [[0.5, 0.5], [0.5, 0.5], [2.0, 2.0], [2.0, 2.0]]
OUT_CHANNELS = [128, 128, 128, 128]

EPOCHS = 250
BATCH_SIZE = 16

train_idx, val_idx, test_idx = get_indices(X_final.shape[0])

train_ds = AutoRegressiveDataset(X_final, train_idx)
val_ds = AutoRegressiveDataset(X_final, val_idx)
test_ds = AutoRegressiveDataset(X_final, test_idx)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

best_model, train_losses, val_mean, val_std = train_model(
    MODES,
    SCALINGS,
    OUT_CHANNELS,
    train_loader,
    val_loader,
    device,
    static_mask=STATIC_MASK,
    epochs=EPOCHS
)

Dataset: Train=2400, Val=300, Test=300
fno_skip='linear'
channel_mlp_skip='linear'
fno_skip='linear'
channel_mlp_skip='linear'
fno_skip='linear'
channel_mlp_skip='linear'
fno_skip='linear'
channel_mlp_skip='linear'
=== START PUSHFORWARD TRAINING | Device: cuda:1 ===
Epoch 001 | LR: 2.00e-04 | Train Loss: 0.002133 | Val L2: 5.46% <--- save
Epoch 002 | LR: 2.00e-04 | Train Loss: 0.000841 | Val L2: 4.77% <--- save
Epoch 003 | LR: 2.00e-04 | Train Loss: 0.000611 | Val L2: 4.63% <--- save
Epoch 004 | LR: 2.00e-04 | Train Loss: 0.000516 | Val L2: 4.26% <--- save
Epoch 005 | LR: 2.00e-04 | Train Loss: 0.000442 | Val L2: 4.07% <--- save
Epoch 006 | LR: 2.00e-04 | Train Loss: 0.000383 | Val L2: 4.08% [Wait 1]
Epoch 007 | LR: 2.00e-04 | Train Loss: 0.000337 | Val L2: 3.70% <--- save
Epoch 008 | LR: 2.00e-04 | Train Loss: 0.000304 | Val L2: 3.68% <--- save
Epoch 009 | LR: 2.00e-04 | Train Loss: 0.000277 | Val L2: 3.69% [Wait 1]
Epoch 010 | LR: 2.00e-04 | Train Loss: 0.000253 | Val L2: 3.54% <--- 

KeyboardInterrupt: 

In [10]:
device = "cuda:3" if torch.cuda.is_available() else "cpu"
MODES = [
    [32, 32], 
    [24, 24], 
    [24, 24], 
    [32, 32]
]

SCALINGS = [
    [0.5, 0.5], 
    [0.5, 0.5], 
    [2.0, 2.0], 
    [2.0, 2.0]
]
OUT_CHANNELS = [128, 128, 128, 128]

EPOCHS = 250
BATCH_SIZE = 16

train_idx, val_idx, test_idx = get_indices(X_final.shape[0])

train_ds = AutoRegressiveDataset(X_final, train_idx)
val_ds = AutoRegressiveDataset(X_final, val_idx)
test_ds = AutoRegressiveDataset(X_final, test_idx)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

model = UNO(
    in_channels=2, 
    out_channels=1,
    domain_padding=0.2,
    hidden_channels=128,
    n_layers=4,
    projection_channels=128,
    uno_out_channels=OUT_CHANNELS,
    uno_n_modes=MODES,
    uno_scalings=SCALINGS,
    channel_mlp_skip="linear"
).to(device)

checkpoint_path = "uno_aboba_fix4_1cycle.pth"
model.load_state_dict(torch.load(checkpoint_path, map_location=device))
model.eval()
print("Модель успешно загружена!")

Dataset: Train=2400, Val=300, Test=300
fno_skip='linear'
channel_mlp_skip='linear'
fno_skip='linear'
channel_mlp_skip='linear'
fno_skip='linear'
channel_mlp_skip='linear'
fno_skip='linear'
channel_mlp_skip='linear'
Модель успешно загружена!


In [ ]:
def generate_rollout_metrics_and_gif(model, data_tensor, test_indices, sample_idx, device, static_mask, save_path="aboba.gif"):
    model.eval()
    mask = static_mask.to(device)
    mask_np = static_mask.cpu().numpy()
    
    idx = test_indices[sample_idx] 
    sample_data = data_tensor[idx].to(device) 
    
    T_steps = sample_data.shape[0]
    Y, X = sample_data.shape[2], sample_data.shape[3]

    true_pressure = sample_data[:, 0, :, :]
    source_wells = sample_data[:, 1, :, :]
    
    # Тензор для предсказаний
    pred_pressure = torch.zeros((T_steps, Y, X), device=device)
    pred_pressure[0] = true_pressure[0] * mask

    print(f"=== Генерация траектории (Сэмпл {idx}, Шагов {T_steps}) ===")

    with torch.no_grad():
        for t in range(0, T_steps - 1):
            curr_p = pred_pressure[t].unsqueeze(0)   # (1, Y, X)
            next_s = source_wells[t+1].unsqueeze(0)  # (1, Y, X)
            
            x_in = torch.stack([curr_p, next_s], dim=1) # (1, 2, Y, X)
            out = model(x_in) # (1, 1, Y, X)
            
            pred_pressure[t+1] = out[0, 0] * mask

    true_np = true_pressure.cpu().numpy()
    pred_np = pred_pressure.cpu().numpy()
    wells_np = source_wells.cpu().numpy()

    global_errors =[]
    well_errors =[]

    for t in range(1, T_steps):
        p_t, p_p = true_np[t], pred_np[t]
        
        w_m = (np.abs(wells_np[t]) > 1e-5).astype(float) * mask_np
        
        g_err = (np.linalg.norm(p_p - p_t) / (np.linalg.norm(p_t) + 1e-8)) * 100
        global_errors.append(g_err)
        
        # Ошибка в скважинах
        if w_m.sum() > 0:
            w_err = (np.linalg.norm((p_p - p_t)*w_m) / (np.linalg.norm(p_t*w_m) + 1e-8)) * 100
            well_errors.append(w_err)

    print(f"Средняя глобальная ошибка (L2): {np.mean(global_errors):.2f}%")
    if well_errors:
        print(f"Средняя ошибка В СКВАЖИНАХ (L2): {np.mean(well_errors):.2f}%")
    else:
        print("В данной траектории нет активных скважин.")

    fig, axes = plt.subplots(1, 4, figsize=(22, 5))
    
    val_min = min(np.min(true_np), np.min(pred_np))
    val_max = max(np.max(true_np), np.max(pred_np))
    cbar_kwargs = {'fraction': 0.046, 'pad': 0.04}
    
    bounds =[0, 1, 3, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 60, 70, 80, 90, 100]
    cmap_error = plt.get_cmap('turbo')
    norm_error = mcolors.BoundaryNorm(bounds, cmap_error.N)
    
    im0 = axes[0].imshow(wells_np[0], cmap='bwr')
    im1 = axes[1].imshow(true_np[0], cmap='jet', vmin=val_min, vmax=val_max)
    im2 = axes[2].imshow(pred_np[0], cmap='jet', vmin=val_min, vmax=val_max)
    im3 = axes[3].imshow(np.zeros_like(true_np[0]), cmap=cmap_error, norm=norm_error)
    
    axes[0].set_title("Скважины (Source)", fontsize=14)
    axes[1].set_title("Истинное давление", fontsize=14)
    axes[2].set_title("Предсказание модели", fontsize=14)
    axes[3].set_title("Процентная ошибка (%)", fontsize=14)
    
    fig.colorbar(im0, ax=axes[0], **cbar_kwargs)
    fig.colorbar(im1, ax=axes[1], **cbar_kwargs)
    fig.colorbar(im2, ax=axes[2], **cbar_kwargs)
    fig.colorbar(im3, ax=axes[3], boundaries=bounds, ticks=bounds, spacing='uniform', **cbar_kwargs)
    
    for ax in axes: 
        ax.set_xticks([])
        ax.set_yticks([])
        
    time_text = fig.suptitle("Инициализация...", fontsize=16, fontweight='bold')

    def update(frame):
        im0.set_array(wells_np[frame])
        im1.set_array(true_np[frame])
        im2.set_array(pred_np[frame])
        
        true_f, pred_f = true_np[frame], pred_np[frame]
        
        masked_true = true_f[mask_np > 0]
        max_t = np.max(np.abs(masked_true)) if len(masked_true) > 0 and np.max(np.abs(masked_true)) > 1e-8 else 1.0
        
        rel_err_map = np.zeros_like(true_f)
        rel_err_map[mask_np > 0] = np.clip((np.abs(true_f - pred_f)[mask_np > 0] / max_t) * 100, 0, 100)
        
        im3.set_array(rel_err_map)
        
        frame_err = (np.linalg.norm(pred_f - true_f) / (np.linalg.norm(true_f) + 1e-8)) * 100
        time_text.set_text(f"Шаг {frame}/{T_steps-1} | Глобальная L2 Ошибка: {frame_err:.2f}%")
        return[im0, im1, im2, im3, time_text]

    ani = animation.FuncAnimation(fig, update, frames=T_steps, interval=200, blit=False)
    ani.save(save_path, writer='pillow', fps=5)
    plt.close(fig)
    print(f"Готово! Гифка сохранена как '{save_path}'")


generate_rollout_metrics_and_gif(
    model=model,
    data_tensor=X_final,
    test_indices=test_idx,
    sample_idx=0,
    device=device,
    static_mask=STATIC_MASK,
    save_path="trajectory_evaluate.gif"
)

=== Генерация траектории (Сэмпл 107, Шагов 21) ===
Средняя глобальная ошибка (L2): 27.47%
Средняя ошибка В СКВАЖИНАХ (L2): 26.51%
Готово! Гифка сохранена как 'trajectory_evaluate.gif'
